In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_7.pickle

In [ ]:
%%cudf.pandas.profile
### cell 7 ###

### cell 7 (optimized for cudf)

# 1) filter part and project key
part_filtered = part[part.P_TYPE == "ECONOMY ANODIZED STEEL"][["P_PARTKEY"]]

# 2) compute volume on lineitem and project
lineitem_filtered = lineitem[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY"]]
lineitem_filtered["VOLUME"] = (
    lineitem.L_EXTENDEDPRICE * (1.0 - lineitem.L_DISCOUNT)
)

# 3) join part → lineitem
total = (
    part_filtered
    .merge(lineitem_filtered,
           left_on="P_PARTKEY", right_on="L_PARTKEY",
           how="inner")[["L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]
)

# 4) join supplier
supplier_filtered = supplier[["S_SUPPKEY", "S_NATIONKEY"]]
total = total.merge(
    supplier_filtered,
    left_on="L_SUPPKEY", right_on="S_SUPPKEY",
    how="inner")[["L_ORDERKEY", "VOLUME", "S_NATIONKEY"]]

# 5) filter orders by date using string literals, extract year
orders_filtered = orders[
    (orders.O_ORDERDATE >= "1995-01-01") &
    (orders.O_ORDERDATE <  "1997-01-01")
]
orders_filtered["O_YEAR"] = orders_filtered.O_ORDERDATE.dt.year
orders_filtered = orders_filtered[["O_ORDERKEY", "O_CUSTKEY", "O_YEAR"]]

total = total.merge(
    orders_filtered,
    left_on="L_ORDERKEY", right_on="O_ORDERKEY",
    how="inner")[["VOLUME", "S_NATIONKEY", "O_CUSTKEY", "O_YEAR"]]

# 6) join customer
customer_filtered = customer[["C_CUSTKEY", "C_NATIONKEY"]]
total = total.merge(
    customer_filtered,
    left_on="O_CUSTKEY", right_on="C_CUSTKEY",
    how="inner")[["VOLUME", "S_NATIONKEY", "O_YEAR", "C_NATIONKEY"]]

# 7) join nation for customer→region and supplier→name
n1 = nation[["N_NATIONKEY", "N_REGIONKEY"]]
n2 = nation[["N_NATIONKEY", "N_NAME"]].rename(columns={"N_NAME": "NATION"})
total = total.merge(n1,
                     left_on="C_NATIONKEY", right_on="N_NATIONKEY",
                     how="inner")[["VOLUME", "S_NATIONKEY", "O_YEAR", "N_REGIONKEY"]]
total = total.merge(n2,
                     left_on="S_NATIONKEY", right_on="N_NATIONKEY",
                     how="inner")[["VOLUME", "O_YEAR", "N_REGIONKEY", "NATION"]]

# 8) filter region
region_filtered = region[region.R_NAME == "AMERICA"][["R_REGIONKEY"]]
total = total.merge(
    region_filtered,
    left_on="N_REGIONKEY", right_on="R_REGIONKEY",
    how="inner")[["VOLUME", "O_YEAR", "NATION"]]

# 9) compute market share without a Python UDF
# total volume per year
total_vol = total.groupby("O_YEAR").VOLUME.sum().reset_index(name="total_volume")
# brazil volume per year
brazil_vol = (
    total[total.NATION == "BRAZIL"]
    .groupby("O_YEAR").VOLUME
    .sum()
    .reset_index(name="brazil_volume")
)
# merge and compute share
total = total_vol.merge(brazil_vol, on="O_YEAR", how="left")
total["MKT_SHARE"] = total.brazil_volume / total.total_volume
# final projection and sort
total = total[["O_YEAR", "MKT_SHARE"]].sort_values("O_YEAR")